# Project 4 starter: shared Setup cells and section skeleton

This notebook is the shared starting point for both tracks (team contract, Section 3). Each of us copies it to our own notebook, trackA.ipynb or trackB.ipynb, and writes only under our own track's headings. On Sun Nov 22 we merge by pasting each track's cells into these same headings, so the final notebook keeps this order.

The Setup section is shared. Neither of us changes it without the other agreeing.

Code follows the official documentation examples of the tools we use. Every code cell starts with a comment that says either Source, naming the documentation page and example the code follows, or Custom, saying why no example fits.

Lines marked DEFAULT are the contract's defaults, kept until the TA confirms or changes them.


## Setup (shared)

In [ ]:
# Custom: shared settings both tracks use (contract Section 3); each track imports its own algorithms in its own cells
import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.datasets import make_circles, make_moons
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 21

# results/ is gitignored (local outputs), figures/ holds report figures, handoff/ holds the files the tracks swap on Nov 19
for folder in ["results", "figures", "handoff"]:
    os.makedirs(folder, exist_ok=True)

print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

### Task 4.1: Preprocessing & Standardization

Spec: "Implement scale normalization using StandardScaler to eliminate feature swamping. Handle skewed features via log-transformation prior to Euclidean distance calculations."

#### Wholesale Customers (Track A)

In [ ]:
# loading wholesale data, 440 clients
wholesale = pd.read_csv("data/Wholesale customers data.csv")
print(wholesale.shape)
wholesale.head()

In [ ]:
# DEFAULT (confirm with TA): Channel and Region are category codes, so they stay out of the distance features
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Source: sklearn FunctionTransformer docs, Examples section: FunctionTransformer(np.log1p)
log_spend = FunctionTransformer(np.log1p).fit_transform(wholesale[SPEND_COLS])

# Source: sklearn StandardScaler docs, fit_transform
X_wholesale = StandardScaler().fit_transform(log_spend)

print("X_wholesale:", X_wholesale.shape)
print("column means:", X_wholesale.mean(axis=0).round(3))
print("column stds: ", X_wholesale.std(axis=0).round(3))

In [ ]:
# Custom: skewness before and after the log, the Task 4.1 evidence for the report (pandas DataFrame.skew)
skew_table = pd.DataFrame({"raw": wholesale[SPEND_COLS].skew(),
                           "after log1p": log_spend.skew()})
skew_table.round(2)

In [ ]:
# Source: matplotlib pyplot.hist; one column per feature, raw spend on top, log1p spend below
fig, axes = plt.subplots(2, len(SPEND_COLS), figsize=(18, 5))
for i, col in enumerate(SPEND_COLS):
    axes[0, i].hist(wholesale[col], bins=30)
    axes[0, i].set_title(col)
    axes[1, i].hist(log_spend[col], bins=30)
axes[0, 0].set_ylabel("raw spend")
axes[1, 0].set_ylabel("log1p(spend)")
fig.suptitle("Wholesale Customers: spending before and after the log transform")
plt.tight_layout()
plt.savefig("figures/wholesale_log_hist.png", dpi=150)
plt.show()

#### Synthetic Geometrical & Anomaly Benchmark (Track B)

DEFAULT (confirm with TA): we generate the 1,500 points ourselves: 700 on two concentric circles, 700 on two moons, and 100 injected noise outliers.

In [ ]:
# Source: sklearn example "Comparing different clustering algorithms on toy datasets": make_circles(factor=0.5, noise=0.05) and make_moons(noise=0.05)
N_CIRCLES, N_MOONS, N_OUTLIERS = 700, 700, 100

X_circles, y_circles = make_circles(n_samples=N_CIRCLES, factor=0.5, noise=0.05, random_state=RANDOM_STATE)
X_moons, y_moons = make_moons(n_samples=N_MOONS, noise=0.05, random_state=RANDOM_STATE)

# Custom: shift the moons right by 3 so the two shapes don't overlap
X_moons = X_moons + [3.0, 0.0]

In [ ]:
# Source: sklearn example "Outlier detection with Local Outlier Factor (LOF)": outliers drawn with np.random.uniform, stacked with np.r_
# Custom: the uniform box is the data's own range plus 0.5, so some outliers land between and inside the shapes (local anomalies)
np.random.seed(RANDOM_STATE)
X_inliers = np.r_[X_circles, X_moons]
X_outliers = np.random.uniform(low=X_inliers.min(axis=0) - 0.5, high=X_inliers.max(axis=0) + 0.5,
                               size=(N_OUTLIERS, 2))

X_synth = np.r_[X_inliers, X_outliers]

# true shape of each point: 0 outer circle, 1 inner circle, 2 and 3 the two moons, -1 injected outlier
y_synth = np.r_[y_circles, y_moons + 2, -np.ones(N_OUTLIERS, dtype=int)]
is_outlier = y_synth == -1

print("X_synth:", X_synth.shape, "| outliers:", is_outlier.sum(), "| points per shape:", np.unique(y_synth, return_counts=True))

### Shared scatter plot

Every cluster scatter plot in the report uses this function, so K-Means, hierarchical, DBSCAN, and LOF plots all share one look. Label -1 (DBSCAN noise, or an outlier) is always a black x.

In [ ]:
# Custom: one scatter style for the whole report; the colours are the palette from the sklearn example
# "Comparing different clustering algorithms on toy datasets", which also draws noise (-1) in black
CLUSTER_COLORS = ["#377eb8", "#ff7f00", "#4daf4a", "#f781bf", "#a65628",
                  "#984ea3", "#999999", "#e41a1c", "#dede00"]


def plot_clusters(X, labels, title, ax=None, filename=None, xlabel="x1", ylabel="x2"):
    """Scatter the first two columns of X coloured by label; -1 is drawn as black x.
    For Wholesale (6 columns) pass a 2D projection, not X_wholesale itself."""
    labels = np.asarray(labels)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    for k in np.unique(labels):
        pts = X[labels == k]
        if k == -1:
            ax.scatter(pts[:, 0], pts[:, 1], s=14, c="black", marker="x", label="noise / outlier")
        else:
            ax.scatter(pts[:, 0], pts[:, 1], s=8, color=CLUSTER_COLORS[k % len(CLUSTER_COLORS)], label=f"cluster {k}")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(markerscale=2, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    if filename:
        ax.figure.savefig(f"figures/{filename}", dpi=150, bbox_inches="tight")
    return ax


plot_clusters(X_synth, y_synth, "Synthetic benchmark: true shapes and injected outliers", filename="synth_truth.png")
plt.show()

### Handoff files (swapped Thu Nov 19)

Each track saves its synthetic-data result with save_handoff, and the other track reads it with pd.read_csv. A file holds one row per point of X_synth, in order, with columns x1, x2, label. A second one-row file holds the silhouette score and how many noise points were left out of it.

Until the real runs exist, the cell below writes placeholder files that use the true shapes as labels and leave the silhouette blank. This lets Track A build the Challenge A side-by-side plot today. The real files overwrite them.

In [ ]:
# Custom: the handoff format agreed in the contract, Section 3
def save_handoff(method, labels, silhouette, n_noise_excluded=0):
    """method is "kmeans" (Track A) or "dbscan" (Track B)."""
    pd.DataFrame({"x1": X_synth[:, 0], "x2": X_synth[:, 1], "label": labels}).to_csv(
        f"handoff/{method}_synth.csv", index=False)
    pd.DataFrame([{"method": method, "silhouette": silhouette, "n_noise_excluded": n_noise_excluded}]).to_csv(
        f"handoff/{method}_summary.csv", index=False)


# placeholders, written only if the real file does not exist yet
for method in ["kmeans", "dbscan"]:
    if not os.path.exists(f"handoff/{method}_synth.csv"):
        save_handoff(method, y_synth, np.nan)

print(pd.read_csv("handoff/dbscan_synth.csv").head(3))
print(pd.read_csv("handoff/dbscan_summary.csv"))

## Task 4.2: K-Means++ Optimization & Silhouette Analysis (Track A)

Spec: "Execute K-Means++ across K ∈ [2, 10]. Plot the Elbow Curve (Inertia/SSE vs. K) and compute Silhouette Coefficients s(i) = (b(i) − a(i)) / max(a(i), b(i)) to select optimal K."

Runs on both datasets. On X_synth it shows K-Means splitting the rings and moons, and at the chosen K Track A calls save_handoff("kmeans", ...).

In [ ]:
# Track A: Task 4.2

## Task 4.3: Hierarchical Linkage Benchmarking (Track A)

Spec: "Apply Agglomerative Hierarchical Clustering using 4 linkage criteria: Single (MIN), Complete (MAX), Average, and Ward's Method. Render dendrogram trees using scipy.cluster.hierarchy."

In [ ]:
# Track A: Task 4.3

## Task 4.4: DBSCAN Density Partitioning (Track B)

Spec: "Construct a K-distance plot (k = MinPts) to identify the 'knee' for optimal Eps parameter selection. Partition data into Core, Border, and Noise points."

Starting points from the contract: MinPts = 4 on X_synth and 12 on X_wholesale. Track B calls save_handoff("dbscan", ...) with the silhouette computed without the noise points.

In [ ]:
# Track B: Task 4.4

## Task 4.5: Anomaly Detection, LOF vs. Global Bounds (Track B)

Spec: "Implement Local Outlier Factor (LOF) and distance-based scoring. Compare LOF local density anomaly scores against global statistical Z-score/Grubbs bounds."

is_outlier is the ground truth for scoring each method. SciPy has no Grubbs test, so it is written from the formula with scipy.stats.t and marked Custom.

In [ ]:
# Track B: Task 4.5

## Report figures for Part I

The written answers to Challenges A, B, and C go in the PDF report. These cells make the figures and worked numbers the answers use.

### Challenge A: side-by-side K-Means vs DBSCAN on the synthetic data (Track A)

Reads handoff/kmeans_synth.csv and handoff/dbscan_synth.csv, and both summary files, for the Silhouette Paradox.

In [ ]:
# Track A: Challenge A figure

### Challenge B: chaining-effect example (Track A)

In [ ]:
# Track A: Challenge B example

### Challenge C: coordinate scenario with LOF ≫ 1 (Track B)

In [ ]:
# Track B: Challenge C example

## Part II: AI Integration (Track B)

The required prompt: an AI writes a Python script for DBSCAN parameter selection and LOF anomaly detection. Its code is pasted here unedited, then audited: did it build a k-distance plot to pick Eps, or guess arbitrary parameters? Every prompt we used is logged in the AI Prompt Appendix.

In [ ]:
# Track B: AI-generated code, unedited